## 🔄 Workflow Overview: From Data to Cost Estimation

This notebook follows a structured pipeline to preprocess the benchmark data and estimate the total evaluation cost using an LLM. Here's an overview of the major steps:

### 1. 📦 Data Preprocessing
- Unzip and load the provided benchmark dataset.
- Parse and extract relevant information such as question text, source code, and mutant variants.
- Organize the data into structured mappings for easy access during prompt construction.

### 2. 🧬 Source–Mutant Mapping
- Identify and map **original source programs** to their corresponding **mutants**.
- Ensure proper alignment between source and mutated code for consistent evaluation.
- Filter or clean mappings if needed (e.g., to remove duplicates or invalid samples).

### 3. ✏️ Prompt Construction
- Generate evaluation prompts using both the **source** and **mutant** code.
- Follow a consistent prompt template to ensure fair comparison across examples.
- Handle few-shot or zero-shot formatting if required by the model.

### 4. 🔢 Token Counting
- Calculate the number of tokens per prompt and completion using the target LLM’s tokenizer.
- Aggregate token counts to compute total tokens for all examples.

### 5. 💸 Cost Estimation
- Multiply total tokens by the **LLM pricing rate** (e.g., cost per 1,000 tokens).
- Separate costs for different models (e.g., GPT-4 vs. GPT-3.5) if applicable.
- Provide breakdown of token and cost statistics across the dataset.

---

This step-by-step process ensures accurate cost estimation while maintaining transparency in how data is transformed and fed into the language model.


### 📁 Define Working and Data Directories

We start by setting up the file system paths required for processing the benchmark data.

- `working_directory`: The root path of the local NTS Evaluation repository.
- `code_extracted_folder_name`: The name of the folder containing the unzipped benchmark files.
- `data_folder`: The full path to the extracted data, used for loading and preprocessing source and mutant files.

These variables will be used throughout the notebook to load and manipulate the dataset.


In [ ]:
working_directory = './NTS'
code_extracted_folder_name = 'extracted'
data_folder = working_directory + '/' + code_extracted_folder_name

### 🗂️ Extract Project Names from Folder Structure

Next, we list the contents of the extracted data folder to identify all available benchmark projects.

- `os.listdir(data_folder)`: Lists all subfolders in the extracted data directory.
- These subfolders typically follow a naming pattern such as:
  - `<project>_source`
  - `<project>_mutants`
  - `<project>_test_suite`

To isolate the base project names, we:
- Strip suffixes like `_source`, `_mutants`, and `_test_suite`
- Store the cleaned names in a `set` to ensure uniqueness

This gives us a list of distinct project names included in the benchmark suite.


In [ ]:
import os

folders = os.listdir(data_folder)
folders
project_names = set([name.replace('_test_suite','').replace('_source','').replace('_mutants','') for name in folders])

In [ ]:
project_names

### 🧬 Map Source Files to Mutant Variants

We now construct a dictionary that maps each project to its original source file and all corresponding mutant files.

#### Code Breakdown:
- For each project name:
  - Locate the original source file inside the `<project>_source/<project>_source/` directory.
  - List all mutant versions inside `<project>_mutants/<project>_mutants/`.
  - For each mutant version, create a full path to the corresponding mutant file (which shares the same filename as the source).
- Store this information in a dictionary `files_map` using the following structure:

```python
files_map = {
    "project_name": (
        "path/to/source/file",
        ["path/to/mutant1", "path/to/mutant2", ...]
    ),
    ...
}
